# 01 Data cleaning (pandas)

**Goal:** profile the 7 curated tables from the pipeline, fix the problems we find, and load a clean
**star schema** into PostgreSQL (`analytics_db`).

**Rules in this notebook**
- Every number in the text below is printed by code. Nothing is typed by hand.
- We *flag* strange values first. We only delete a row when it is clearly wrong.
- Each decision has: the problem, the rows affected, what we did, and why.

**Input:** `data/raw/pipeline_output/` (Parquet files from the Spark job).
**Output:** `data/processed/*.parquet`, the star schema in `analytics_db`, and two small summary files in `reports/`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.db import get_engine, run_sql_file, copy_df  # noqa: E402

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

RAW = ROOT / "data" / "raw" / "pipeline_output"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
(ROOT / "reports").mkdir(exist_ok=True)

TABLES = ["orders", "order_items", "payments", "reviews", "customers", "products", "sellers"]


def load(table):
    files = sorted((RAW / table).glob("*.parquet"))
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)


raw = {t: load(t) for t in TABLES}
decisions = []  # one row per decision, saved at the end


def decision(n, title, problem, rows, action, why):
    rows = int(rows)
    decisions.append({"decision": n, "title": title, "rows_affected": rows, "action": action})
    display(Markdown(
        f"**Decision {n}: {title}**\n\n- Problem: {problem}\n- Rows affected: **{rows:,}**\n"
        f"- Action: {action}\n- Why: {why}"
    ))


print({t: len(df) for t, df in raw.items()})

{'orders': 99441, 'order_items': 112650, 'payments': 103886, 'reviews': 99224, 'customers': 99441, 'products': 32951, 'sellers': 3095}


## 1. Profile each table

In [2]:
KEYS = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
}


def profile_row(t, df):
    return {
        "table": t,
        "rows": len(df),
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "key": " + ".join(KEYS[t]),
        "duplicate_keys": int(df.duplicated(KEYS[t]).sum()),
    }


profile_before = pd.DataFrame([profile_row(t, df) for t, df in raw.items()])
profile_before

,table,rows,columns,missing_cells,duplicate_rows,key,duplicate_keys
0,orders,99441,20,13934,0,order_id,0
1,order_items,112650,12,0,0,order_id + order_item_id,0
2,payments,103886,5,0,0,order_id + payment_sequential,0
3,reviews,99224,5,0,0,review_id + order_id,0
4,customers,99441,5,0,0,customer_id,0
5,products,32951,5,1222,0,product_id,0
6,sellers,3095,3,0,0,seller_id,0


In [3]:
# Data types of every table
for t, df in raw.items():
    print(f"--- {t}: " + ", ".join(f"{c} ({str(d)})" for c, d in df.dtypes.items()))

--- orders: order_id (str), customer_id (str), order_status (str), purchase_ts (datetime64[ns]), approved_ts (datetime64[ns]), carrier_ts (datetime64[ns]), delivered_ts (datetime64[ns]), estimated_delivery_ts (datetime64[ns]), customer_unique_id (str), city (str), state (str), item_count (float64), items_value (float64), freight_value (float64), payment_value (float64), max_installments (float64), main_payment_type (str), review_score (float64), delivery_days (float64), delay_days (float64)
--- order_items: order_id (str), order_item_id (int32), product_id (str), seller_id (str), customer_id (str), customer_unique_id (str), customer_state (str), category_en (str), order_status (str), purchase_ts (datetime64[ns]), price (float64), freight_value (float64)
--- payments: order_id (str), payment_sequential (int32), payment_type (str), payment_installments (int32), payment_value (float64)
--- reviews: review_id (str), order_id (str), review_score (int32), creation_ts (datetime64[ns]), answer

In [4]:
# Missing values per column (only columns with at least one missing value)
rows = []
for t, df in raw.items():
    for c, n in df.isna().sum().items():
        if n:
            rows.append({"table": t, "column": c, "missing": int(n), "pct": round(100 * n / len(df), 2)})
pd.DataFrame(rows)

,table,column,missing,pct
0,orders,approved_ts,160,0.16
1,orders,carrier_ts,1783,1.79
2,orders,delivered_ts,2965,2.98
3,orders,item_count,775,0.78
4,orders,items_value,775,0.78
5,orders,freight_value,775,0.78
6,orders,payment_value,1,0.00
7,orders,max_installments,1,0.00
8,orders,main_payment_type,1,0.00
9,orders,review_score,768,0.77


Reading the profile: all keys are unique and there are no duplicate rows, so the Spark job already removed duplicates.
The missing values are mostly *real*, not errors. For example, an order that was never delivered has no delivery date.
The next sections check each problem one by one.

## 2. Orders: status, items and dates

In [5]:
o = raw["orders"].copy()
items = raw["order_items"].copy()
pay = raw["payments"].copy()
rev = raw["reviews"].copy()
cust = raw["customers"].copy()
prod = raw["products"].copy()
sell = raw["sellers"].copy()

# Referential checks: every child row must have a parent
checks = {
    "items without an order": (~items.order_id.isin(o.order_id)).sum(),
    "payments without an order": (~pay.order_id.isin(o.order_id)).sum(),
    "reviews without an order": (~rev.order_id.isin(o.order_id)).sum(),
    "orders without a customer": (~o.customer_id.isin(cust.customer_id)).sum(),
    "items without a product": (~items.product_id.isin(prod.product_id)).sum(),
    "items without a seller": (~items.seller_id.isin(sell.seller_id)).sum(),
}
pd.Series(checks, name="orphan rows").to_frame()

,orphan rows
items without an order,0
payments without an order,0
reviews without an order,0
orders without a customer,0
items without a product,0
items without a seller,0


In [6]:
print("Order status counts:")
display(o.order_status.value_counts().rename("orders").to_frame())

no_items = ~o.order_id.isin(items.order_id)
print("Orders without any item, by status:")
display(o[no_items].order_status.value_counts().rename("orders").to_frame())

Order status counts:


,orders
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


Orders without any item, by status:


,orders
order_status,
unavailable,603
canceled,164
created,5
invoiced,2
shipped,1


In [7]:
o["has_items"] = ~no_items
o["item_count"] = o["item_count"].fillna(0).astype("int64")
o[["items_value", "freight_value"]] = o[["items_value", "freight_value"]].fillna(0.0)

decision(
    1, "Orders without items",
    "Some orders have no row in order_items, so they have no price. Most are canceled or unavailable.",
    no_items.sum(),
    "Kept the orders. Set item_count, items_value and freight_value to 0 and added the flag has_items = False.",
    "Deleting them would hide real cancellations. The flag lets every revenue query leave them out.",
)

# A valid sale = has items and was not canceled or unavailable
o["is_valid_sale"] = o.has_items & ~o.order_status.isin(["canceled", "unavailable"])
excluded = ~o.is_valid_sale
display(pd.crosstab(o.order_status, o.is_valid_sale, margins=True).rename(columns={False: "not valid", True: "valid sale"}))

decision(
    2, "Which orders count as a sale",
    "Canceled and unavailable orders were never fulfilled, so their value should not count as revenue.",
    excluded.sum(),
    "Added the flag is_valid_sale (has items and status is not canceled or unavailable). "
    "Orders in other states (created, approved, invoiced, processing, shipped) stay valid because they are real, paid-for orders.",
    "One clear, simple rule for revenue. The rule is stored in the data, so SQL, Excel and Power BI use the same one.",
)

**Decision 1: Orders without items**

- Problem: Some orders have no row in order_items, so they have no price. Most are canceled or unavailable.
- Rows affected: **775**
- Action: Kept the orders. Set item_count, items_value and freight_value to 0 and added the flag has_items = False.
- Why: Deleting them would hide real cancellations. The flag lets every revenue query leave them out.

is_valid_sale,not valid,valid sale,All
order_status,,,
approved,0,2,2
canceled,625,0,625
created,5,0,5
delivered,0,96478,96478
invoiced,2,312,314
processing,0,301,301
shipped,1,1106,1107
unavailable,609,0,609
All,1242,98199,99441


**Decision 2: Which orders count as a sale**

- Problem: Canceled and unavailable orders were never fulfilled, so their value should not count as revenue.
- Rows affected: **1,242**
- Action: Added the flag is_valid_sale (has items and status is not canceled or unavailable). Orders in other states (created, approved, invoiced, processing, shipped) stay valid because they are real, paid-for orders.
- Why: One clear, simple rule for revenue. The rule is stored in the data, so SQL, Excel and Power BI use the same one.

In [8]:
# Delivery dates
delivered = o.order_status.eq("delivered")
bad_delivered = delivered & o.delivered_ts.isna()
ts_not_delivered = ~delivered & o.delivered_ts.notna()
print("delivered before purchase:", int((o.delivered_ts < o.purchase_ts).sum()))
print("delivered status but no delivery date:", int(bad_delivered.sum()))
print("delivery date but status is not delivered:", int(ts_not_delivered.sum()))
display(o[ts_not_delivered].order_status.value_counts().rename("orders").to_frame())

# Recompute delivery metrics in pandas, only for clean delivered orders
ok = delivered & o.delivered_ts.notna()
dd = (o.delivered_ts - o.purchase_ts).dt.total_seconds() / 86400
dl = (o.delivered_ts - o.estimated_delivery_ts).dt.total_seconds() / 86400
o["delivery_days"] = dd.where(ok).round(2)
o["delay_days"] = dl.where(ok).round(2)
o["is_late"] = (o.delay_days > 0).astype("boolean").where(o.delay_days.notna())

# The pandas result must match the Spark result for the rows we keep
spark_dd = raw["orders"]["delivery_days"].where(ok)
print("max difference to Spark delivery_days:", float((o.delivery_days - spark_dd).abs().max()))

decision(
    3, "Delivery date does not match the status",
    "Orders marked delivered without a delivery date, and orders with a delivery date but a different status.",
    bad_delivered.sum() + ts_not_delivered.sum(),
    "Kept the rows. delivery_days, delay_days and is_late are filled only when the status is delivered AND the delivery date exists.",
    "A delivery time is only meaningful for orders that really arrived. The rest stays empty, not guessed.",
)

delivered before purchase: 0
delivered status but no delivery date: 8
delivery date but status is not delivered: 6


,orders
order_status,
canceled,6


max difference to Spark delivery_days: 0.010000000000005116


**Decision 3: Delivery date does not match the status**

- Problem: Orders marked delivered without a delivery date, and orders with a delivery date but a different status.
- Rows affected: **14**
- Action: Kept the rows. delivery_days, delay_days and is_late are filled only when the status is delivered AND the delivery date exists.
- Why: A delivery time is only meaningful for orders that really arrived. The rest stays empty, not guessed.

In [9]:
# Dates in the wrong order
carrier_before_purchase = o.carrier_ts < o.purchase_ts
delivered_before_carrier = o.delivered_ts < o.carrier_ts
o["flag_ts_inconsistent"] = (carrier_before_purchase | delivered_before_carrier).fillna(False)
print("carrier date before purchase:", int(carrier_before_purchase.sum()))
print("delivered before carrier date:", int(delivered_before_carrier.sum()))

decision(
    4, "Dates in the wrong order",
    "For some orders the carrier pick-up date is before the purchase date, or the delivery date is before the carrier date.",
    o.flag_ts_inconsistent.sum(),
    "Kept the rows and added the flag flag_ts_inconsistent. No date was changed.",
    "The purchase date and the delivery date are still correct (no delivery is before the purchase), and these two are the "
    "dates the analysis uses. The carrier date is not used.",
)

carrier date before purchase: 166
delivered before carrier date: 23


**Decision 4: Dates in the wrong order**

- Problem: For some orders the carrier pick-up date is before the purchase date, or the delivery date is before the carrier date.
- Rows affected: **189**
- Action: Kept the rows and added the flag flag_ts_inconsistent. No date was changed.
- Why: The purchase date and the delivery date are still correct (no delivery is before the purchase), and these two are the dates the analysis uses. The carrier date is not used.

## 3. Payments and reviews

In [10]:
print("Payment types:")
display(pay.payment_type.value_counts().rename("rows").to_frame())
zero_pay = pay.payment_value <= 0
bad_inst = pay.payment_installments < 1
display(pay[zero_pay].payment_type.value_counts().rename("zero-value rows by type").to_frame())
print("installments below 1:", int(bad_inst.sum()), " (types:", pay[bad_inst].payment_type.unique().tolist(), ")")

Payment types:


,rows
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


,zero-value rows by type
payment_type,
voucher,6
not_defined,3


installments below 1: 2  (types: ['credit_card'] )


In [11]:
pay["payment_installments"] = pay.payment_installments.clip(lower=1)
pay["payment_value"] = pay.payment_value.round(2)
pay["flag_zero_value"] = zero_pay

decision(
    5, "Installments below 1",
    "A payment cannot have 0 installments. One payment is one installment at least.",
    bad_inst.sum(),
    "Set the installments to 1.",
    "It is a clear data entry error and the fix does not change any money amount.",
)
decision(
    6, "Payments with value 0",
    "Some payments have a value of 0 (mostly vouchers, plus the type not_defined).",
    zero_pay.sum(),
    "Kept them. The order value is the sum of its payments, so a 0 payment adds nothing.",
    "They are real payment lines, and removing them would break the link between order and payments.",
)

# Order level payment fields from the cleaned payments
pay_agg = pay.groupby("order_id").agg(
    payment_value=("payment_value", "sum"), max_installments=("payment_installments", "max")
).reset_index()
pay_agg["payment_value"] = pay_agg.payment_value.round(2)
o = o.drop(columns=["payment_value", "max_installments"]).merge(pay_agg, on="order_id", how="left")

**Decision 5: Installments below 1**

- Problem: A payment cannot have 0 installments. One payment is one installment at least.
- Rows affected: **2**
- Action: Set the installments to 1.
- Why: It is a clear data entry error and the fix does not change any money amount.

**Decision 6: Payments with value 0**

- Problem: Some payments have a value of 0 (mostly vouchers, plus the type not_defined).
- Rows affected: **9**
- Action: Kept them. The order value is the sum of its payments, so a 0 payment adds nothing.
- Why: They are real payment lines, and removing them would break the link between order and payments.

In [12]:
dup_rev = rev.duplicated("order_id", keep=False)
print("orders with more than one review:", int(rev.loc[dup_rev, "order_id"].nunique()))
print("review score counts:")
display(rev.review_score.value_counts().sort_index().rename("reviews").to_frame())

rev_sorted = rev.sort_values(["order_id", "answer_ts", "creation_ts"])
rev_clean = rev_sorted.drop_duplicates("order_id", keep="last").reset_index(drop=True)
dropped_reviews = len(rev) - len(rev_clean)
assert rev_clean.review_score.between(1, 5).all()

decision(
    7, "More than one review for an order",
    "Some orders have several reviews. This would count the same order twice in review statistics.",
    dropped_reviews,
    "Kept one review per order: the most recently answered one. The order-level review_score now comes from this review.",
    "One order should have one score. The last review is the customer's final opinion.",
)

o = o.drop(columns=["review_score"]).merge(rev_clean[["order_id", "review_score"]], on="order_id", how="left")
print("orders without a review (score stays empty):", int(o.review_score.isna().sum()))

orders with more than one review: 547
review score counts:


,reviews
review_score,
1,11424
2,3151
3,8179
4,19142
5,57328


**Decision 7: More than one review for an order**

- Problem: Some orders have several reviews. This would count the same order twice in review statistics.
- Rows affected: **551**
- Action: Kept one review per order: the most recently answered one. The order-level review_score now comes from this review.
- Why: One order should have one score. The last review is the customer's final opinion.

orders without a review (score stays empty): 768


## 4. Products, customers and text

In [13]:
unknown_cat = prod.category_en.eq("unknown")
print("products without a category:", int(unknown_cat.sum()))
print("products without a weight:", int(prod.weight_g.isna().sum()))

decision(
    8, "Products without a category",
    "Some products have no category name, so they cannot be grouped in category reports.",
    unknown_cat.sum(),
    "Category stays 'unknown' (set by the Spark job). Missing weight and photo counts stay empty.",
    "'unknown' shows the real size of the gap in the Pareto and category charts. Guessing a category would be invention.",
)

# Text checks
for name, df, cols in [("customers", cust, ["city", "state"]), ("sellers", sell, ["city", "state"])]:
    for c in cols:
        df[c] = df[c].str.strip()
print("customer states:", cust.state.nunique(), "| seller states:", sell.state.nunique())
print("customer_id rows:", len(cust), "| real persons (customer_unique_id):", cust.customer_unique_id.nunique())

products without a category: 610
products without a weight: 2


**Decision 8: Products without a category**

- Problem: Some products have no category name, so they cannot be grouped in category reports.
- Rows affected: **610**
- Action: Category stays 'unknown' (set by the Spark job). Missing weight and photo counts stay empty.
- Why: 'unknown' shows the real size of the gap in the Pareto and category charts. Guessing a category would be invention.

customer states: 27 | seller states: 23
customer_id rows: 99441 | real persons (customer_unique_id): 96096


In [14]:
decision(
    9, "One person has many customer_id values",
    "Olist creates a new customer_id for every order. The real person is customer_unique_id.",
    len(cust) - cust.customer_unique_id.nunique(),
    "dim_customer has one row per customer_unique_id (city, state and zip of the latest order). "
    "fact_orders keeps both ids. All repeat, cohort and RFM logic will use customer_unique_id.",
    "If we used customer_id, nearly every customer would look like a first-time buyer.",
)

**Decision 9: One person has many customer_id values**

- Problem: Olist creates a new customer_id for every order. The real person is customer_unique_id.
- Rows affected: **3,345**
- Action: dim_customer has one row per customer_unique_id (city, state and zip of the latest order). fact_orders keeps both ids. All repeat, cohort and RFM logic will use customer_unique_id.
- Why: If we used customer_id, nearly every customer would look like a first-time buyer.

## 5. Outliers (IQR rule: flag first, then decide)

In [15]:
def iqr_fence(s):
    q1, q3 = s.quantile([0.25, 0.75])
    return q3 + 1.5 * (q3 - q1)


price_fence = iqr_fence(items.price)
freight_fence = iqr_fence(items.freight_value)
order_fence = iqr_fence(o.loc[o.is_valid_sale, "items_value"])

items["flag_price_outlier"] = items.price > price_fence
o["flag_value_outlier"] = o.is_valid_sale & (o.items_value > order_fence)

summary = pd.DataFrame({
    "upper fence": [price_fence, freight_fence, order_fence],
    "max value": [items.price.max(), items.freight_value.max(), o.loc[o.is_valid_sale, "items_value"].max()],
    "rows above fence": [
        int(items.flag_price_outlier.sum()),
        int((items.freight_value > freight_fence).sum()),
        int(o.flag_value_outlier.sum()),
    ],
}, index=["item price", "item freight", "order items value"]).round(2)
display(summary)

print("Categories of the price outliers (top 5):")
display(items.loc[items.flag_price_outlier, "category_en"].value_counts().head(5).rename("outlier items").to_frame())

,upper fence,max value,rows above fence
item price,277.40,6735.00,8427
item freight,33.25,409.68,11613
order items value,305.90,13440.00,7851


Categories of the price outliers (top 5):


,outlier items
category_en,
health_beauty,1157
watches_gifts,975
sports_leisure,485
auto,425
cool_stuff,390


In [16]:
decision(
    10, "Price and order value outliers",
    "Some prices and order values are far above the IQR upper fence. Prices have a long right tail.",
    items.flag_price_outlier.sum(),
    "Flagged only (flag_price_outlier, flag_value_outlier). No row was removed or changed.",
    "The high prices are in categories where expensive goods are normal. They are real sales, so removing them would "
    "understate revenue. Later analysis can report with and without outliers.",
)

**Decision 10: Price and order value outliers**

- Problem: Some prices and order values are far above the IQR upper fence. Prices have a long right tail.
- Rows affected: **8,427**
- Action: Flagged only (flag_price_outlier, flag_value_outlier). No row was removed or changed.
- Why: The high prices are in categories where expensive goods are normal. They are real sales, so removing them would understate revenue. Later analysis can report with and without outliers.

## 6. Time coverage: which months are complete?

The dataset does not cover all months equally. We check the monthly orders and define an *analysis period*.

In [17]:
monthly = o[o.is_valid_sale].groupby(o.purchase_ts.dt.to_period("M")).size().rename("valid orders").to_frame()
display(monthly)

,valid orders
purchase_ts,
2016-09,2
2016-10,290
2016-12,1
2017-01,787
2017-02,1718
2017-03,2617
2017-04,2377
2017-05,3640
2017-06,3205


In [18]:
PERIOD_START, PERIOD_END = "2017-01", "2018-08"
ym = o.purchase_ts.dt.strftime("%Y-%m")
outside = ~ym.between(PERIOD_START, PERIOD_END)
print("valid orders outside the analysis period:", int((o.is_valid_sale & outside).sum()),
      "of", int(o.is_valid_sale.sum()))

decision(
    11, "Incomplete months at the start and end",
    "Orders in 2016 are very few, one month is missing (2016-11), and 2018-09 and 2018-10 have almost no orders. "
    "These months would look like a collapse in sales.",
    (o.is_valid_sale & outside).sum(),
    f"Kept all rows. dim_date has the flag in_analysis_period for the full months {PERIOD_START} to {PERIOD_END}.",
    "Trends, growth and the forecast must use full months only, or the first and last months will give wrong results.",
)

valid orders outside the analysis period: 294 of 98199


**Decision 11: Incomplete months at the start and end**

- Problem: Orders in 2016 are very few, one month is missing (2016-11), and 2018-09 and 2018-10 have almost no orders. These months would look like a collapse in sales.
- Rows affected: **294**
- Action: Kept all rows. dim_date has the flag in_analysis_period for the full months 2017-01 to 2018-08.
- Why: Trends, growth and the forecast must use full months only, or the first and last months will give wrong results.

## 7. Build the star schema tables

In [19]:
# dim_date: every day from the first to the last date in the orders table
date_cols = ["purchase_ts", "approved_ts", "carrier_ts", "delivered_ts", "estimated_delivery_ts"]
start = o[date_cols].min().min().normalize()
end = o[date_cols].max().max().normalize()
d = pd.date_range(start, end, freq="D")
dim_date = pd.DataFrame({"full_date": d})
dim_date["date_key"] = d.year * 10000 + d.month * 100 + d.day
dim_date["year"] = d.year
dim_date["quarter"] = d.quarter
dim_date["month"] = d.month
dim_date["month_name"] = d.month_name()
dim_date["year_month"] = d.strftime("%Y-%m")
dim_date["day"] = d.day
dim_date["day_of_week"] = d.dayofweek + 1
dim_date["day_name"] = d.day_name()
dim_date["week_of_year"] = d.isocalendar().week.astype(int).values
dim_date["is_weekend"] = d.dayofweek >= 5
dim_date["in_analysis_period"] = (dim_date.year_month >= PERIOD_START) & (dim_date.year_month <= PERIOD_END)
dim_date["full_date"] = dim_date.full_date.dt.date
dim_date = dim_date[["date_key", "full_date", "year", "quarter", "month", "month_name", "year_month", "day",
                     "day_of_week", "day_name", "week_of_year", "is_weekend", "in_analysis_period"]]
print("dim_date:", dim_date.full_date.min(), "to", dim_date.full_date.max(), "|", len(dim_date), "days")

# dim_customer: one row per real person, with the city/state/zip of the latest order
last = (o.sort_values("purchase_ts").drop_duplicates("customer_unique_id", keep="last")
        [["customer_unique_id", "customer_id", "city", "state"]]
        .merge(cust[["customer_id", "zip_code_prefix"]], on="customer_id", how="left")
        .drop(columns="customer_id"))
dim_customer = last.reset_index(drop=True)

dim_product = prod[["product_id", "category_pt", "category_en", "weight_g", "photos_qty"]].copy()
dim_seller = sell[["seller_id", "city", "state"]].copy()

# fact_orders
o["purchase_date"] = o.purchase_ts.dt.date
o["purchase_date_key"] = (o.purchase_ts.dt.year * 10000 + o.purchase_ts.dt.month * 100 + o.purchase_ts.dt.day)
o = o.rename(columns={"city": "customer_city", "state": "customer_state"})
fact_orders = o[[
    "order_id", "customer_id", "customer_unique_id", "customer_state", "customer_city", "order_status",
    "purchase_ts", "purchase_date", "purchase_date_key", "approved_ts", "carrier_ts", "delivered_ts",
    "estimated_delivery_ts", "item_count", "items_value", "freight_value", "payment_value", "max_installments",
    "main_payment_type", "review_score", "delivery_days", "delay_days", "is_late", "has_items", "is_valid_sale",
    "flag_ts_inconsistent", "flag_value_outlier",
]].copy()
fact_orders["items_value"] = fact_orders.items_value.round(2)
fact_orders["freight_value"] = fact_orders.freight_value.round(2)
fact_orders["max_installments"] = fact_orders.max_installments.astype("Int64")
fact_orders["review_score"] = fact_orders.review_score.astype("Int64")

fact_order_items = items.assign(
    purchase_date_key=items.purchase_ts.dt.year * 10000 + items.purchase_ts.dt.month * 100 + items.purchase_ts.dt.day,
    price=items.price.round(2),
    freight_value=items.freight_value.round(2),
)[["order_id", "order_item_id", "product_id", "seller_id", "customer_unique_id", "purchase_date_key",
   "order_status", "price", "freight_value", "flag_price_outlier"]]

fact_payments = pay[["order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"]]
fact_reviews = rev_clean[["order_id", "review_id", "review_score", "creation_ts", "answer_ts"]]
dim_product["photos_qty"] = dim_product.photos_qty.astype("Int64")

clean = {
    "dim_date": dim_date, "dim_customer": dim_customer, "dim_product": dim_product, "dim_seller": dim_seller,
    "fact_orders": fact_orders, "fact_order_items": fact_order_items, "fact_payments": fact_payments,
    "fact_reviews": fact_reviews,
}

# Checks before loading: keys are unique and every foreign key has a parent
assert fact_orders.order_id.is_unique and dim_customer.customer_unique_id.is_unique
assert not fact_order_items.duplicated(["order_id", "order_item_id"]).any()
assert fact_order_items.order_id.isin(fact_orders.order_id).all()
assert fact_payments.order_id.isin(fact_orders.order_id).all()
assert fact_reviews.order_id.isin(fact_orders.order_id).all()
assert fact_orders.customer_unique_id.isin(dim_customer.customer_unique_id).all()
assert fact_orders.purchase_date_key.isin(dim_date.date_key).all()
print({k: len(v) for k, v in clean.items()})

dim_date: 2016-09-04 to 2018-11-12 | 800 days


{'dim_date': 800, 'dim_customer': 96096, 'dim_product': 32951, 'dim_seller': 3095, 'fact_orders': 99441, 'fact_order_items': 112650, 'fact_payments': 103886, 'fact_reviews': 98673}


## 8. Data quality summary: before and after

In [20]:
def n_missing(df):
    return int(df.isna().sum().sum())


pairs = [
    ("orders", "fact_orders", ["order_id"]),
    ("order_items", "fact_order_items", ["order_id", "order_item_id"]),
    ("payments", "fact_payments", ["order_id", "payment_sequential"]),
    ("reviews", "fact_reviews", ["order_id"]),
    ("customers", "dim_customer", ["customer_unique_id"]),
    ("products", "dim_product", ["product_id"]),
    ("sellers", "dim_seller", ["seller_id"]),
]
rows = []
for before_name, after_name, key in pairs:
    b, a = raw[before_name], clean[after_name]
    rows.append({
        "raw table": before_name, "clean table": after_name,
        "rows before": len(b), "rows after": len(a),
        "missing cells before": n_missing(b), "missing cells after": n_missing(a),
        "duplicate keys before": int(b.duplicated(KEYS[before_name]).sum()),
        "duplicate keys after": int(a.duplicated(key).sum()),
        "columns before": b.shape[1], "columns after": a.shape[1],
    })
dq = pd.DataFrame(rows)
dq.to_csv(ROOT / "reports" / "data_quality_summary.csv", index=False)
dq

,raw table,clean table,rows before,rows after,missing cells before,missing cells after,duplicate keys before,duplicate keys after,columns before,columns after
0,orders,fact_orders,99441,99441,13934,14592,0,0,20,27
1,order_items,fact_order_items,112650,112650,0,0,0,0,12,10
2,payments,fact_payments,103886,103886,0,0,0,0,5,5
3,reviews,fact_reviews,99224,98673,0,0,0,0,5,5
4,customers,dim_customer,99441,96096,0,0,0,0,5,4
5,products,dim_product,32951,32951,1222,1222,0,0,5,5
6,sellers,dim_seller,3095,3095,0,0,0,0,3,3


**How to read this table**
- *Rows after* is lower than *rows before* only where we removed rows on purpose (reviews: one per order; customers: one per real person).
- *Missing cells after* can be higher because new columns were added (for example `delivery_days` is empty for orders that were not delivered).
  These empty values are real, not errors.

In [21]:
dec = pd.DataFrame(decisions)
dec.to_csv(ROOT / "reports" / "data_cleaning_decisions.csv", index=False)
dec

,decision,title,rows_affected,action
0,1,Orders without items,775,"Kept the orders. Set item_count, items_value a..."
1,2,Which orders count as a sale,1242,Added the flag is_valid_sale (has items and st...
2,3,Delivery date does not match the status,14,"Kept the rows. delivery_days, delay_days and i..."
3,4,Dates in the wrong order,189,Kept the rows and added the flag flag_ts_incon...
4,5,Installments below 1,2,Set the installments to 1.
5,6,Payments with value 0,9,Kept them. The order value is the sum of its p...
6,7,More than one review for an order,551,Kept one review per order: the most recently a...
7,8,Products without a category,610,Category stays 'unknown' (set by the Spark job...
8,9,One person has many customer_id values,3345,dim_customer has one row per customer_unique_i...
9,10,Price and order value outliers,8427,"Flagged only (flag_price_outlier, flag_value_o..."


## 9. Save to Parquet and load into PostgreSQL (`analytics_db`)

In [22]:
for name, df in clean.items():
    df.to_parquet(OUT / f"{name}.parquet", index=False)
print("saved", len(clean), "parquet files in", OUT.relative_to(ROOT))

engine = get_engine()
run_sql_file(engine, ROOT / "sql" / "00_star_schema.sql")
for name in ["dim_date", "dim_customer", "dim_product", "dim_seller",
             "fact_orders", "fact_order_items", "fact_payments", "fact_reviews"]:
    copy_df(engine, clean[name], name)
print("loaded all tables")

saved 8 parquet files in data\processed


loaded all tables


In [23]:
from sqlalchemy import text

with engine.connect() as c:
    db_counts = {n: c.execute(text(f"select count(*) from {n}")).scalar() for n in clean}
    n_pk = c.execute(text("select count(*) from information_schema.table_constraints "
                          "where table_schema='public' and constraint_type='PRIMARY KEY'")).scalar()
    n_fk = c.execute(text("select count(*) from information_schema.table_constraints "
                          "where table_schema='public' and constraint_type='FOREIGN KEY'")).scalar()
    n_ix = c.execute(text("select count(*) from pg_indexes where schemaname='public' and indexname like 'ix_%'")).scalar()
    sql_value = c.execute(text("select round(sum(items_value),2) from fact_orders where is_valid_sale")).scalar()

comparison = pd.DataFrame({"pandas rows": {k: len(v) for k, v in clean.items()}, "database rows": db_counts})
comparison["match"] = comparison["pandas rows"] == comparison["database rows"]
display(comparison)
print(f"primary keys: {n_pk}, foreign keys: {n_fk}, extra indexes: {n_ix}")

pandas_value = round(float(fact_orders.loc[fact_orders.is_valid_sale, "items_value"].sum()), 2)
print("items value of valid sales, pandas:", pandas_value, "| database:", float(sql_value))
assert comparison.match.all()
assert abs(pandas_value - float(sql_value)) < 0.01

,pandas rows,database rows,match
dim_date,800,800,True
dim_customer,96096,96096,True
dim_product,32951,32951,True
dim_seller,3095,3095,True
fact_orders,99441,99441,True
fact_order_items,112650,112650,True
fact_payments,103886,103886,True
fact_reviews,98673,98673,True


primary keys: 8, foreign keys: 8, extra indexes: 9
items value of valid sales, pandas: 13494400.74 | database: 13494400.74


In [24]:
# A first look at the clean data in the database
pd.read_sql(text('''
    select o.order_id, o.order_status, o.purchase_date, o.items_value, o.freight_value,
           o.main_payment_type, o.review_score, o.delivery_days, o.is_late, o.is_valid_sale, c.state
    from fact_orders o join dim_customer c using (customer_unique_id)
    order by o.purchase_ts limit 5'''), engine)

,order_id,order_status,purchase_date,items_value,freight_value,main_payment_type,review_score,delivery_days,is_late,is_valid_sale,state
0,2e7a8482f6fb09756ca50c10d7bfc047,shipped,2016-09-04,72.89,63.34,credit_card,1.0,NaN,None,True,RR
1,e5fa5a7210941f7d56d0208e4e071d35,canceled,2016-09-05,59.50,15.56,credit_card,1.0,NaN,None,False,RS
2,809a282bbd5dbcabb6f2f724fca862ec,canceled,2016-09-13,0.00,0.00,credit_card,1.0,NaN,None,False,SP
3,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15,134.97,8.49,NaN,1.0,54.81,True,True,SP
4,71303d7e93b399f5bcd537d124c0bcfa,canceled,2016-10-02,100.00,9.34,credit_card,1.0,NaN,None,False,SP


## 10. Summary

- All 7 curated tables were profiled. Keys are unique and the data has no exact duplicates.
- 11 decisions are written above (and saved in `reports/data_cleaning_decisions.csv`). Rows were removed only for
  duplicate reviews. Everything else is flagged or left empty on purpose.
- The star schema is in `analytics_db`: `dim_date`, `dim_customer`, `dim_product`, `dim_seller`, `fact_orders`,
  `fact_order_items`, `fact_payments`, `fact_reviews`, with primary keys, foreign keys and indexes.
- Business rules stored in the data: `is_valid_sale`, `in_analysis_period`, `is_late`, and the outlier flags.
- Revenue means the item price without freight (`items_value`). Freight and total payment are kept as separate columns.